# Mimicking Finance 复现（公司数据版）— 逐配置对照

一个 holdings parquet 就够，**不需要 WRDS**。`Run All` 跑完全部配置。
每个配置跑完自动 `R.free()` 清缓存（20 核机器连跑五个不会 OOM）。

## 已在真实 WRDS 数据上验证过的基准（1094 万行、12321 只基金、2010–2024）

| | 论文 | 我们实测 |
|---|---|---|
| precision（真实持仓） | — | 0.5755 (gbm) / 0.5291 (lstm) |
| precision（含 padding, N=75） | **0.71** | **0.7156** |
| naive（含 padding） | **0.52** | **0.5208** |
| **Table X Q5−Q1 (tradeable)** | **−0.79** (t=−3.05) | **−0.660 (gbm) / −0.657 (lstm)** |
| Table XII Q1−Q5（同期口径） | +1.06 (t=5.74) | +1.213 (t=5.46) |

**LSTM 和 GBM 给出几乎相同的经济结论**（−0.657 vs −0.660），尽管 LSTM 精度更低
（0.529 vs 0.576）。所以结论由**数据和时间口径**决定，不由架构决定——这一点是跑出来的，
不是推的。

## 三种架构

| `model` | 是什么 | 速度 |
|---|---|---|
| `gbm` | 梯度提升树，序列拍平成 `y_lag1..4` 等列 | 最快，CPU 几分钟 |
| `lstm` | 权重共享单仓位序列：一个样本 = 一个仓位最近 8 季 `[T, F]` | 中等 |
| `panel_lstm` | **论文原架构**：一个样本 = 一个 fund-quarter 整截面<br>`[T, N, F] → LSTM(N·F→numcell) → [N, 3]` | 最慢 |

`panel_lstm` 会额外打印 **padding 占比**——就是把 precision 从 0.58 垫到 0.71 的那部分。

## 特征

`volume` 和 `n_holdings` 已接入。成交量相关的五个特征只用 t 期及以前的信息：

| 特征 | 含义 |
|---|---|
| `log_volume` | 成交量水平 |
| `vol_rank` | 每季度横截面分位数——**对"volume 是股数还是金额"免疫** |
| `pos_to_vol` | `shares / volume`，几个单位成交量才能卖完这个仓位 |
| `d_log_vol` | 成交量季度变化（精确 t−1，跨缺口不借值） |
| `amihud` | `|收益| / 成交量`，Amihud 式非流动性 |

`pos_to_vol` 区分的是**"想不想交易"和"能不能交易"**：占了 20 倍成交量的仓位，
经理即使想清仓也只能分季度慢慢减。这比论文那个"是否连续持有 8 季"的
feasibility mask 更贴近真实交易摩擦。

文件里没有 `volume` 列会自动跳过，不影响运行。

## 关键开关 `use_manager_memory`

`fs_hold_rate`（"这个经理从来不动这个仓位"）能把精度推高，但它把
**"交易方向可预测"** 偷换成 **"这个基金根本不交易"**。低换手基金历史上跑赢，
于是 **Table X 反号**（实测 −0.660 → +0.116）。

**复现论文用 `False`**；`True` 只是反面教材。

## 三种时间口径

`accuracy(t)` 要看到 `shares[t+1]` 才知道 → **t+1 才知道**；13F 季末后 45–60 天才公开
→ **t+2 才可交易**。

- `contemporaneous` acc(t) × t→t+1　**有重叠、有偏**
- `predictive`　　　 acc(t) × t+1→t+2　无重叠但忽略披露延迟
- `tradeable`　　　　acc(t) × t+2→t+3　**真正可交易**

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from dataclasses import replace
%load_ext autoreload
%autoreload 2
import company_replication as R
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

## 0. 基础配置

改 `data_path`；列名不同就改 `col_map`（左边是你文件里的列名）。

In [ ]:
BASE = R.Config(
    data_path = "manager_holdings/master_batches_return_filtered/master_all_funds_add_filter_ivy_rank_active_rank.parquet",
    inv_type_codes = (401,),
    max_rank   = 25,           # 也是 panel_lstm 的 N
    template_N = 75,           # 算 padding 用的模板宽度（论文例子是 75/100）
    min_years  = 7, min_holdings = 10,
    window_q   = 28, test_q = 8, step = 8,
    drop_missing_position = True,
    # --- 神经网络专用；序列按索引惰性拼装，默认全量不抽样 ---
    seq_len = 8, hidden = 64, dropout = 0.25, lr = 3e-3,
    max_epochs = 25, patience = 5, batch = 8192, device = "auto",
    lstm_max_train = None,     # 20 核 CPU 上嫌慢可设 300_000（拿精度换时间）
    lstm_max_rows  = None,
)
BASE

## 1. 建面板（只做一次，所有配置共用）

**先核对这里的类别分布。** 我们 WRDS 数据是「卖 0.386 / 持 0.292 / 买 0.322」，
其中 23.6% 是**精确为零**的股数变化。你的 hold 比例若远低于此（如 5%），
说明 `future_1q_shares_change_pct` 口径不同，后面所有数要重新校准。

In [ ]:
panel = R.load_and_prepare(BASE)
print(f"\n面板 {len(panel):,} 行 | {panel.fund.nunique():,} 基金 | {panel.qi.max()+1} 季度")
RESULTS = {}

### 检查成交量特征的单位

**`pos_to_vol` 的分布要看一眼。** 如果 `volume` 是**成交金额**而 `shares` 是**股数**，
这个比值就没有物理意义（作为单调变换可能仍有预测力，但解释不了）。

判断方法：`pos_to_vol` 的中位数如果是 10⁻⁶ 这种极小的数，说明分母是金额、量纲对不上，
这时候把它换成 `position_value / volume` 才是正确的"几天能卖完"。
`vol_rank` 和 `amihud` 不受影响。

In [ ]:
volf = [c for c in ("log_volume","vol_rank","pos_to_vol","d_log_vol","amihud")
        if c in panel.columns]
if volf:
    display(panel[volf].describe().loc[["count","mean","std","min","25%","50%","75%","max"]].round(4))
    med = panel["pos_to_vol"].median()
    print(f"pos_to_vol 中位数 = {med:.6g}")
    print("→ 量纲合理（volume 是股数）" if med > 1e-3 else
          "→ ⚠️ 极小，volume 可能是金额；考虑改用 position_value / volume")
else:
    print("没有 volume 列，成交量特征已跳过")
print(f"\n实际用到的特征 {len([f for f in BASE.features if f in panel.columns])} 个")

In [ ]:
CONFIGS = {
    "A_gbm_no_mem":    dict(model="gbm",        use_manager_memory=False),
    "B_gbm_mem":       dict(model="gbm",        use_manager_memory=True),
    "C_lstm_no_mem":   dict(model="lstm",       use_manager_memory=False),
    "D_lstm_mem":      dict(model="lstm",       use_manager_memory=True),
    "E_panel_no_mem":  dict(model="panel_lstm", use_manager_memory=False),
}
CONFIGS

---
# A — gbm，无 manager memory　（最快的基线，先跑这个）

WRDS 实测：accuracy 0.5755，Table X Q5−Q1 (tradeable) = **−0.660 (t=−3.20)**。

In [ ]:
RESULTS["A_gbm_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["A_gbm_no_mem"]), "A_gbm_no_mem")
R.free(RESULTS)          # 清掉 preds，避免连跑多个配置 OOM

In [ ]:
r = RESULTS["A_gbm_no_mem"]
display(r["precision_table"].round(4))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table X / {tm} --");   display(r["tableX"][tm].round(3))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table XII / {tm} --"); display(r["tableXII"][tm].round(3))

---
# B — gbm，**有** manager memory　（反面教材）

预期 Table X **反号**。WRDS 实测：−0.660 → **+0.116 (t=+0.82)**。

In [ ]:
RESULTS["B_gbm_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["B_gbm_mem"]), "B_gbm_mem")
R.free(RESULTS)          # 清掉 preds，避免连跑多个配置 OOM

In [ ]:
r = RESULTS["B_gbm_mem"]
display(r["precision_table"].round(4))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table X / {tm} --");   display(r["tableX"][tm].round(3))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table XII / {tm} --"); display(r["tableXII"][tm].round(3))

---
# C — lstm，无 manager memory　**（论文的架构）**

论文用的是 LSTM，所以这一栏才是架构层面的复现。
WRDS 实测：accuracy 0.5291，Table X Q5−Q1 (tradeable) = **−0.657 (t=−3.33)**
——与 gbm 的 −0.660 几乎一致。

20 核 CPU 上会比 gbm 慢不少（GPU 上约 2 分钟/窗口）。嫌慢设 `lstm_max_train=300_000`。

In [ ]:
RESULTS["C_lstm_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["C_lstm_no_mem"]), "C_lstm_no_mem")
R.free(RESULTS)          # 清掉 preds，避免连跑多个配置 OOM

In [ ]:
r = RESULTS["C_lstm_no_mem"]
display(r["precision_table"].round(4))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table X / {tm} --");   display(r["tableX"][tm].round(3))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table XII / {tm} --"); display(r["tableXII"][tm].round(3))

---
# D — lstm，有 manager memory

In [ ]:
RESULTS["D_lstm_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["D_lstm_mem"]), "D_lstm_mem")
R.free(RESULTS)          # 清掉 preds，避免连跑多个配置 OOM

In [ ]:
r = RESULTS["D_lstm_mem"]
display(r["precision_table"].round(4))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table X / {tm} --");   display(r["tableX"][tm].round(3))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table XII / {tm} --"); display(r["tableXII"][tm].round(3))

---
# E — panel_lstm　**（论文原始 `(T, N, F) → (N×3)` 整截面架构）**

一个样本 = 一个 fund-quarter 的**整个截面**：第 j 列是该基金在 t 期排名第 j 的证券，
沿时间回溯同一只证券。基金持仓不足 N 时多出来的列就是 **padding**。

这一栏会打印 **padding 占比**——直接看到把 precision 垫高的是多少。
最慢的一个；内存不够就把 `max_rank` 调小（N 越小张量越小）。

In [ ]:
RESULTS["E_panel_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["E_panel_no_mem"]), "E_panel_no_mem")
R.free(RESULTS)          # 清掉 preds，避免连跑多个配置 OOM

In [ ]:
r = RESULTS["E_panel_no_mem"]
display(r["precision_table"].round(4))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table X / {tm} --");   display(r["tableX"][tm].round(3))
for tm in ("tradeable","predictive","contemporaneous"):
    print(f"-- Table XII / {tm} --"); display(r["tableXII"][tm].round(3))

---
# 汇总

`X与论文同号 = 是` 表示该配置在 **tradeable** 口径下复现了论文 Table X 的方向
（越难预测的基金越跑赢）。预期：**无 manager memory 的配置都是「是」**。

In [ ]:
summary = R.summarize(RESULTS)
summary

In [ ]:
cmp = pd.DataFrame([
    {"指标":"precision (含padding)",     "论文":0.71,
     **{k: round(v["precision_table"].iloc[2]["precision"],4) for k,v in RESULTS.items()}},
    {"指标":"naive (含padding)",         "论文":0.52,
     **{k: round(v["precision_table"].iloc[2]["naive"],4) for k,v in RESULTS.items()}},
    {"指标":"Table X Q5-Q1 (tradeable)", "论文":-0.79,
     **{k: round(v["tableX"]["tradeable"].iloc[-1].CRET_0_4,3) for k,v in RESULTS.items()}},
    {"指标":"Table XII Q1-Q5 (同期)",    "论文":1.06,
     **{k: round(v["tableXII"]["contemporaneous"].iloc[-1].mean_qret,3) for k,v in RESULTS.items()}},
])
cmp

## 存档

In [ ]:
import os
os.makedirs("outputs_company", exist_ok=True)
for tag, r in RESULTS.items():
    r["precision_table"].to_csv(f"outputs_company/precision_{tag}.csv", index=False)
    for tm in ("tradeable","predictive","contemporaneous"):
        r["tableX"][tm].to_csv(f"outputs_company/tableX_{tag}_{tm}.csv", index=False)
        r["tableXII"][tm].to_csv(f"outputs_company/tableXII_{tag}_{tm}.csv", index=False)
summary.to_csv("outputs_company/summary.csv", index=False)
cmp.to_csv("outputs_company/compare_with_paper.csv", index=False)
print("已保存到 outputs_company/")

## 内存不够时

20 核机器上如果某个配置 OOM：

```python
# 1) 只保留结果表，彻底丢掉预测明细
R.free(RESULTS)

# 2) 单独重跑那个配置，缩小规模
c = replace(BASE, model="panel_lstm", max_rank=15)     # N 越小张量越小
c = replace(BASE, model="lstm", lstm_max_train=300_000) # 每窗口训练量封顶
c = replace(BASE, model="lstm", lstm_max_rows=2_000_000) # 建序列前先抽样面板

# 3) 实在不行只跑 gbm，经济结论一样
```

各配置的内存量级（1000 万行面板）：`gbm` ~3 GB，`lstm` ~2 GB（惰性索引），
`panel_lstm` 随 `max_rank` 增长。